In [1]:
import pandas as pd
import duckdb
import numpy as np
import os

# 1. KONFIGURASI PATH

In [2]:
RAW_PATH   = "/content/superstore_powerbi.xlsx "
DB_PATH    = "/content/superstore.duckdb"
OUT_CSV    = "/content/cleaned_superstore_fact.csv"
OUT_DIR    = "/content"
EXCEL_PATH = f"{OUT_DIR}/superstore_powerbi_full.xlsx"

# 2. DATA CLEANING & FEATURE ENGINEERING

In [3]:
print("\n[1/4] Memuat dan Membersihkan Data...")

# Logika fallback jika dieksekusi di Colab dengan file di root directory
if not os.path.exists(RAW_PATH):
    if os.path.exists("superstore_powerbi.xlsx"):
        RAW_PATH = "superstore_powerbi.xlsx"
    else:
        raise FileNotFoundError(f"File mentah tidak ditemukan di: {RAW_PATH}")

df = pd.read_excel(RAW_PATH)

# Standardisasi nama kolom (snake_case)
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[/ ]", "_", regex=True)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# Parsing tanggal
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"]  = pd.to_datetime(df["ship_date"], errors="coerce")

# Menghapus data tanpa tanggal pesanan
df.dropna(subset=["order_date"], inplace=True)

# Ekstraksi komponen waktu & durasi pengiriman
df["order_year"]       = df["order_date"].dt.year
df["order_month"]      = df["order_date"].dt.month
df["order_quarter"]    = df["order_date"].dt.quarter
df["order_month_name"] = df["order_date"].dt.strftime("%b")
df["ship_days"]        = (df["ship_date"] - df["order_date"]).dt.days

# Validasi kolom numerik
for col in ["sales", "quantity", "discount", "profit"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Normalisasi variabel profit
df["profit_clean"] = df["profit"]
df["profit_margin"] = (df["profit_clean"] / df["sales"]).replace([np.inf, -np.inf], 0).round(4)
df["revenue_per_qty"] = (df["sales"] / df["quantity"].replace(0, 1)).round(2)

# Klasifikasi Profit Band
def profit_segment(p):
    if pd.isna(p): return "Unknown"
    if p < 0:      return "Loss"
    elif p < 50:   return "Low Profit"
    elif p < 200:  return "Mid Profit"
    else:          return "High Profit"
df["profit_band"] = df["profit_clean"].apply(profit_segment)

# Klasifikasi Discount Bucket
def categorize_discount(d):
    if pd.isna(d) or d == 0: return 'No Discount'
    elif d <= 0.10: return '1-10%'
    elif d <= 0.20: return '11-20%'
    elif d <= 0.30: return '21-30%'
    else: return 'Above 30%'
df["discount_bucket"] = df["discount"].apply(categorize_discount)

# Menyimpan data bersih
df.to_csv(OUT_CSV, index=False)
print(f"      ✓ Data bersih selesai ({len(df):,} baris). Tersimpan di {OUT_CSV}")


[1/4] Memuat dan Membersihkan Data...
      ✓ Data bersih selesai (10,194 baris). Tersimpan di /content/cleaned_superstore_fact.csv


# 3. DUCKDB SQL AGGREGATIONS

In [4]:
print("\n[2/4] Memproses Analisis SQL via DuckDB...")

con = duckdb.connect(DB_PATH)
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM df")

# Agregasi 1: Sales by Year & Region
by_yr_reg = con.execute("""
    SELECT order_year, region,
           COUNT(DISTINCT order_id) AS total_orders, COUNT(*) AS total_items,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit, AVG(discount) AS avg_discount
    FROM orders GROUP BY order_year, region
""").df()
by_yr_reg["profit_margin_pct"] = (by_yr_reg["total_profit"] / by_yr_reg["total_sales"] * 100).round(1)

# Agregasi 2: Sales by Category / Sub-Category
by_cat = con.execute("""
    SELECT category, subcategory,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit,
           SUM(quantity) AS total_qty, COUNT(DISTINCT order_id) AS total_orders
    FROM orders GROUP BY category, subcategory
""").df()
by_cat["profit_margin_pct"] = (by_cat["total_profit"] / by_cat["total_sales"] * 100).round(1)

# Agregasi 3: Monthly Trend
by_month = con.execute("""
    SELECT order_year, order_quarter, order_month, order_month_name,
           SUM(sales) AS monthly_sales, SUM(profit_clean) AS monthly_profit, COUNT(DISTINCT order_id) AS order_count
    FROM orders GROUP BY order_year, order_quarter, order_month, order_month_name
    ORDER BY order_year, order_month
""").df()

# Agregasi 4: Customer Segment
by_seg = con.execute("""
    SELECT segment, COUNT(DISTINCT customer_id) AS unique_customers, COUNT(DISTINCT order_id) AS total_orders,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit
    FROM orders GROUP BY segment
""").df()
by_seg["profit_margin_pct"] = (by_seg["total_profit"] / by_seg["total_sales"] * 100).round(1)
by_seg["avg_sales_per_customer"] = (by_seg["total_sales"] / by_seg["unique_customers"]).round(0)

# Agregasi 5: Discount Impact
by_disc = con.execute("""
    SELECT discount_bucket, COUNT(order_id) AS order_count, AVG(sales) AS avg_sales, AVG(profit_clean) AS avg_profit,
           SUM(profit_clean) AS total_profit, SUM(sales) AS total_sales
    FROM orders GROUP BY discount_bucket
""").df()
by_disc["profit_margin_pct"] = (by_disc["total_profit"] / by_disc["total_sales"] * 100).round(1)

# Agregasi 6: Shipping
by_ship = con.execute("""
    SELECT ship_mode, AVG(ship_days) AS avg_ship_days, COUNT(DISTINCT order_id) AS total_orders,
           SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit
    FROM orders GROUP BY ship_mode ORDER BY avg_ship_days
""").df()
by_ship["profit_margin_pct"] = (by_ship["total_profit"] / by_ship["total_sales"] * 100).round(1)

# Agregasi 7: Top States
by_state = con.execute("""
    SELECT state_province, region, SUM(sales) AS total_sales, SUM(profit_clean) AS total_profit,
           COUNT(DISTINCT order_id) AS total_orders
    FROM orders GROUP BY state_province, region ORDER BY total_sales DESC
""").df()
by_state["profit_margin_pct"] = (by_state["total_profit"] / by_state["total_sales"] * 100).round(1)

# Agregasi 8: Loss-Making Orders (RCA)
by_loss_making = con.execute("""
    SELECT order_id, customer_name, segment, state_province, category, subcategory, product_name,
           ROUND(sales, 2) AS sales, ROUND(profit_clean, 2) AS profit, ROUND(discount * 100, 0) AS discount_pct, profit_band
    FROM orders WHERE profit_clean < 0 ORDER BY profit_clean ASC LIMIT 500
""").df()

# Agregasi 9: Customer RFM Value
by_rfm = con.execute("""
    SELECT customer_id, customer_name, segment, COUNT(DISTINCT order_id) AS frequency,
           ROUND(SUM(sales), 2) AS monetary, ROUND(SUM(profit_clean), 2) AS total_profit,
           MAX(order_date) AS last_order_date, ROUND(SUM(sales) / COUNT(DISTINCT order_id), 0) AS avg_order_value
    FROM orders GROUP BY customer_id, customer_name, segment ORDER BY monetary DESC
""").df()
by_rfm["last_order_date"] = by_rfm["last_order_date"].astype(str)

con.close()
print("      ✓ Ekstraksi seluruh agregasi SQL selesai.")


[2/4] Memproses Analisis SQL via DuckDB...
      ✓ Ekstraksi seluruh agregasi SQL selesai.


# 4. EXPORT EXCEL UNTUK POWER BI

In [5]:
print("\n[3/4] Mengekspor Data ke Excel (Power BI Format)...")

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    df_export = df.copy()
    df_export["order_date"] = df_export["order_date"].astype(str)
    df_export["ship_date"]  = df_export["ship_date"].astype(str)

    # Export semua sheet ke Excel
    df_export.to_excel(writer, sheet_name="Orders_Fact", index=False)
    by_yr_reg.to_excel(writer, sheet_name="Sales_Year_Region", index=False)
    by_cat.to_excel(writer, sheet_name="Category_Performance", index=False)
    by_month.to_excel(writer, sheet_name="Monthly_Trend", index=False)
    by_seg.to_excel(writer, sheet_name="Segment_Analysis", index=False)
    by_disc.to_excel(writer, sheet_name="Discount_Impact", index=False)
    by_ship.to_excel(writer, sheet_name="Shipping_Analysis", index=False)
    by_state.to_excel(writer, sheet_name="State_Performance", index=False)
    by_loss_making.to_excel(writer, sheet_name="Loss_Making_Orders", index=False)
    by_rfm.to_excel(writer, sheet_name="Customer_RFM_Value", index=False)

print(f"      ✓ Excel mult-sheet tersimpan di {EXCEL_PATH}")


[3/4] Mengekspor Data ke Excel (Power BI Format)...
      ✓ Excel mult-sheet tersimpan di /content/superstore_powerbi_full.xlsx


# 5. BUSINESS INSIGHTS SUMMARY

In [6]:
print("\n[4/4] Ringkasan Eksekutif Bisnis")
print("=" * 70)

def fmt_m(v): return f"${v/1e6:.1f}M"

total_sales  = df["sales"].sum()
total_profit = df["profit_clean"].sum()
total_orders = df["order_id"].nunique()

print(f"📊 OVERALL PERFORMANCE")
print(f"   Revenue     : ${total_sales/1e6:.2f}M")
print(f"   Profit      : ${total_profit/1e6:.2f}M ({total_profit/total_sales*100:.1f}% margin)")
print(f"   Orders      : {total_orders:,}")

print(f"\n📦 CATEGORY BREAKDOWN")
cat_s = df.groupby("category").agg(s=("sales","sum"), p=("profit_clean","sum"))
for cat, row in cat_s.iterrows():
    print(f"   - {cat:<20} Sales: {fmt_m(row.s):<10} Margin: {row.p/row.s*100:.1f}%")

print(f"\n🏆 TOP 3 STATES BY REVENUE")
top3 = by_state.head(3)
for _, r in top3.iterrows():
    print(f"   - {r['state_province']:<20} ${r['total_sales']/1e6:.2f}M  (Margin {r['profit_margin_pct']:.1f}%)")

print(f"\n🏷️ DISCOUNT IMPACT SUMMARY")
for _, r in by_disc.iterrows():
    icon = "✅" if r["profit_margin_pct"] >= 0 else "❌"
    print(f"   {icon} {str(r['discount_bucket']):<15} Margin: {r['profit_margin_pct']:.1f}%")

print("\n" + "=" * 70)
print(" ✅ PIPELINE COMPLETE!")
print(f" 📂 Data Bersih (CSV): {OUT_CSV}")
print(f" 🗄️ Database DuckDB  : {DB_PATH}")
print(f" 📊 Excel Power BI   : {EXCEL_PATH}")
print("=" * 70)


[4/4] Ringkasan Eksekutif Bisnis
📊 OVERALL PERFORMANCE
   Revenue     : $1011.45M
   Profit      : $310.17M (30.7% margin)
   Orders      : 5,111

📦 CATEGORY BREAKDOWN
   - Furniture            Sales: $521.7M    Margin: -2.3%
   - Office Supplies      Sales: $212.4M    Margin: 64.5%
   - Technology           Sales: $277.4M    Margin: 66.7%

🏆 TOP 3 STATES BY REVENUE
   - Texas                $228.83M  (Margin -19.2%)
   - California           $225.81M  (Margin 73.4%)
   - New York             $109.08M  (Margin 100.1%)

🏷️ DISCOUNT IMPACT SUMMARY
   ✅ 11-20%          Margin: 75.2%
   ✅ 1-10%           Margin: 112.0%
   ❌ Above 30%       Margin: -42.9%
   ✅ No Discount     Margin: 127.7%
   ❌ 21-30%          Margin: -65.6%

 ✅ PIPELINE COMPLETE!
 📂 Data Bersih (CSV): /content/cleaned_superstore_fact.csv
 🗄️ Database DuckDB  : /content/superstore.duckdb
 📊 Excel Power BI   : /content/superstore_powerbi_full.xlsx
